# Manual Tests for the Data Pebbles SDK and Backend

## Prerequisites
- Make sure the data pebbles server is running and accessible.

## Tests

In [1]:
import sys

sys.path.append("/Users/leon/Desktop/programming/data-pebbles/sdk")

from src.data_pebbles.sdk import DataPebbles

dp = DataPebbles("http://localhost:8000")
projects = dp.projects.list_projects()
projects

[]

### 1 · Create a test project

In [2]:
project_id = dp.projects.create_project(
	"transform_test1",
	description="Manual test of bronze→silver→gold pipeline",
)
print(project_id)

1


### 2 · Bronze — create resource and upload raw CSV

In [3]:
import io

import polars as pl

bronze_id = dp.bronze.create_resource("raw_sales", project_id=project_id)
print("Created bronze resource:", bronze_id)

# Inline sample data: product / quantity / price
csv_data = (
	b"product,quantity,price\n"
	b"Apple,100,1.50\n"
	b"Banana,200,0.80\n"
	b"Cherry,50,3.00\n"
	b"Date,30,5.00\n"
)
upload_result = dp.bronze.upload(bronze_id, data=csv_data, file_name="raw_sales.csv")
print("Upload result:", upload_result)

Created bronze resource: 1
Upload result: {'message': 'File uploaded successfully.'}


In [4]:
# Inspect what we just uploaded
raw_bytes = dp.bronze.download(bronze_id)
print("Raw bronze data:")
print(pl.read_csv(io.BytesIO(raw_bytes)))

print("\nVersions:", dp.bronze.list_versions(bronze_id))

Raw bronze data:
shape: (4, 3)
┌─────────┬──────────┬───────┐
│ product ┆ quantity ┆ price │
│ ---     ┆ ---      ┆ ---   │
│ str     ┆ i64      ┆ f64   │
╞═════════╪══════════╪═══════╡
│ Apple   ┆ 100      ┆ 1.5   │
│ Banana  ┆ 200      ┆ 0.8   │
│ Cherry  ┆ 50       ┆ 3.0   │
│ Date    ┆ 30       ┆ 5.0   │
└─────────┴──────────┴───────┘

Versions: [VersionResponse(id=1, resource_id=1, version=1, status='archived', s3_key='bronze/raw_sales_20260318T192837_1e4bb9cdded646eeb1ac9901bbe064bd.csv', created_at='2026-03-18T19:28:37.199547+00:00', updated_at='2026-03-18T19:28:37.199556+00:00', additional_properties={})]


### 3 · Silver — `silver_transform`: divide all numeric columns by 10

In [5]:
silver_id = dp.silver.create_resource("clean_sales", project_id=project_id)
print("Created silver resource:", silver_id)


@dp.silver_transform(target_id=silver_id, from_bronze_id=bronze_id)
def scale_down(lf: pl.LazyFrame) -> pl.LazyFrame:
	"""Divide all numeric columns by 10 (bronze → silver transform)."""
	num_cols = [col for col, dtype in lf.schema.items() if dtype.is_numeric()]
	return lf.with_columns([pl.col(c) / 10 for c in num_cols])


scale_down()
print("\nSilver data after transform:")
print(dp.silver.download(silver_id).collect())

print("\nSilver versions:", dp.silver.list_versions(silver_id))

Created silver resource: 1

Silver data after transform:


/var/folders/h6/c2yy64s57ks3xp5rvf8z16t00000gn/T/ipykernel_9586/2795369332.py:8: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  num_cols = [col for col, dtype in lf.schema.items() if dtype.is_numeric()]


shape: (4, 3)
┌─────────┬──────────┬───────┐
│ product ┆ quantity ┆ price │
│ ---     ┆ ---      ┆ ---   │
│ str     ┆ f64      ┆ f64   │
╞═════════╪══════════╪═══════╡
│ Cherry  ┆ 5.0      ┆ 0.3   │
│ Banana  ┆ 20.0     ┆ 0.08  │
│ Apple   ┆ 10.0     ┆ 0.15  │
│ Date    ┆ 3.0      ┆ 0.5   │
└─────────┴──────────┴───────┘

Silver versions: [SilverLineageResponse(id=1, resource_id=1, delta_version=0, from_resource_id=1, created_at='2026-03-18T19:28:37.402291+00:00', additional_properties={})]


### 4 · Gold — `gold_transform`: aggregate sums of the silver data

In [6]:
gold_id = dp.gold.create_resource("sales_summary", project_id=project_id)
print("Created gold resource:", gold_id)


@dp.gold_transform(target_id=gold_id, from_silver_ids=[silver_id])
def summarize(sources: dict[int, pl.LazyFrame]) -> pl.LazyFrame:
	"""Sum all numeric columns across the silver dataset."""
	lf = sources[silver_id]
	num_cols = [col for col, dtype in lf.schema.items() if dtype.is_numeric()]
	return lf.select([pl.col(c).sum() for c in num_cols])


summarize()
print("\nGold data after aggregation:")
print(dp.gold.download(gold_id).collect())

print("\nGold versions:", dp.gold.list_versions(gold_id))

Created gold resource: 1

Gold data after aggregation:
shape: (1, 2)
┌──────────┬───────┐
│ quantity ┆ price │
│ ---      ┆ ---   │
│ f64      ┆ f64   │
╞══════════╪═══════╡
│ 38.0     ┆ 1.03  │
└──────────┴───────┘

Gold versions: [GoldLineageResponse(id=1, resource_id=1, delta_version=0, from_resource_id=1, created_at='2026-03-18T19:28:37.675877+00:00', additional_properties={})]


/var/folders/h6/c2yy64s57ks3xp5rvf8z16t00000gn/T/ipykernel_9586/483314025.py:9: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  num_cols = [col for col, dtype in lf.schema.items() if dtype.is_numeric()]


### 5 · Cleanup — delete test resources and project

In [7]:
# Run this cell to tear down everything created above
dp.gold.delete_resource(gold_id)
dp.silver.delete_resource(silver_id)
dp.bronze.delete_resource(bronze_id)
dp.projects.delete_project(project_id)
print("All test resources and project deleted.")

All test resources and project deleted.
